In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. Load Subscription CDC Work Table
# ============================================================

subscription_cdc_df = spark.table(
    "customer360_dev.control.subscription_cdc_work"
)


# ============================================================
# 2. Get Current / Successful Run ID
# ============================================================

run_id_row = (
    subscription_cdc_df
    .select("_pipeline_run_id")
    .filter(
        F.col("_pipeline_run_id").isNotNull()
    )
    .first()
)

last_run_id = (
    run_id_row["_pipeline_run_id"]
    if run_id_row
    else None
)

print(
    "Successful Subscription Run ID:",
    last_run_id
)

Successful Subscription Run ID: None


In [0]:
# ============================================================
# 3. Get Latest Processed Timestamp and File
# ============================================================

latest_processed = (
    subscription_cdc_df
    .agg(
        F.max("updated_at").alias(
            "last_processed_value"
        ),
        F.max("_source_file").alias(
            "last_processed_file"
        )
    )
    .first()
)

last_processed_value = (
    latest_processed["last_processed_value"]
)

last_processed_file = (
    latest_processed["last_processed_file"]
)

print(
    "Last processed timestamp:",
    last_processed_value
)

print(
    "Last processed file:",
    last_processed_file
)

Last processed timestamp: None
Last processed file: None


In [0]:
# ============================================================
# 4. Reconciliation Status
# ============================================================

reconciliation_status = "PASS"

In [0]:
# ============================================================
# 5. Build Watermark Update Record
# ============================================================

if reconciliation_status == "PASS":

    watermark_update_df = (
        spark.createDataFrame(
            [
                (
                    "subscriptions",
                    "SUBSCRIPTION_PLATFORM",
                    last_processed_value,
                    last_processed_file,
                    last_run_id,
                    "SUCCESS"
                )
            ],
            """
            entity_name STRING,
            source_system STRING,
            last_processed_value TIMESTAMP,
            last_processed_file STRING,
            last_run_id STRING,
            load_status STRING
            """
        )
        .withColumn(
            "updated_at",
            F.current_timestamp()
        )
    )

    watermark_update_df.createOrReplaceTempView(
        "subscription_watermark_update"
    )

else:

    print(
        "Subscription reconciliation failed. "
        "Watermark will NOT be updated."
    )

In [0]:
# ============================================================
# 6. Merge Into Watermark Table
# ============================================================

if reconciliation_status == "PASS":

    spark.sql("""
        MERGE INTO customer360_dev.control.watermark AS tgt

        USING subscription_watermark_update AS src

        ON tgt.entity_name = src.entity_name

        WHEN MATCHED THEN
        UPDATE SET
            tgt.source_system =
                src.source_system,

            tgt.last_processed_value =
                src.last_processed_value,

            tgt.last_processed_file =
                src.last_processed_file,

            tgt.last_run_id =
                src.last_run_id,

            tgt.load_status =
                src.load_status,

            tgt.updated_at =
                src.updated_at

        WHEN NOT MATCHED THEN
        INSERT (
            entity_name,
            source_system,
            last_processed_value,
            last_processed_file,
            last_run_id,
            load_status,
            updated_at
        )

        VALUES (
            src.entity_name,
            src.source_system,
            src.last_processed_value,
            src.last_processed_file,
            src.last_run_id,
            src.load_status,
            src.updated_at
        )
    """)

    print(
        "Subscription watermark updated successfully."
    )

Subscription watermark updated successfully.
